
# ASTERIX `.ast` notebook: clean message parsing, FSPEC extraction, and UAP mapping

This notebook is designed as a **structured learning notebook** for ASTERIX binary files. It does **not** try to be a full decoder yet. Instead, it focuses on the parts that must be understood first if you want to decode `.ast` files correctly and later convert them into clean tabular data.

## What an ASTERIX file really contains

An ASTERIX file is typically a **continuous sequence of message blocks**. There is no special end marker. The parser moves forward block by block using the header of each message:

- **Octet 1** → `CAT` (category)
- **Octets 2-3** → `LEN` (total length of the ASTERIX block, including `CAT` and `LEN`)
- **Remaining octets** → one or more **records**

That means the file structure is conceptually:

```text
[CAT][LEN][record area][CAT][LEN][record area][CAT][LEN][record area]...
```

## Why the FSPEC is central

Inside a record, the first structure is the **FSPEC** (*Field Specification*). The FSPEC does **not** contain the data values themselves. Its job is to announce **which fields are present** in that record.

- Each FSPEC octet contributes **7 FRN flags**.
- The last bit is **FX**.
- If `FX = 1`, the FSPEC continues into the next octet.
- Once the full FSPEC is known, the **UAP** (*User Application Profile*) is used to translate each `FRN` into the corresponding **Data Item** for that category.

## Important practical limitation

A single ASTERIX message may contain **more than one record**. However, after the FSPEC, the record body contains a mix of:

- fixed-length items
- variable-length items
- repetitive items
- compound items
- expansion fields (`RE`, `SP`)

Because of that, the start of **record #2** cannot be found safely unless **record #1 is decoded completely**. For that reason, this notebook works on the **first record only** when it parses the FSPEC and maps FRNs to Data Items.

## Notebook workflow

1. **Load the raw binary stream**
2. **Split the stream into ASTERIX message blocks using `CAT + LEN`**
3. **Parse the first-record FSPEC of each message and list the detected FRNs**
4. **Map those FRNs to Data Items using the CAT021 and CAT048 UAP tables**

The code is intentionally organized around a few clean pandas tables:

- `messages_df` → one row per ASTERIX message block
- `fspec_df` → one row per message with the first-record FSPEC and FRN list
- `uap_df` → normalized UAP lookup table across categories
- `data_items_df` → one row per detected `(message, FRN, Data Item)` pair

The goal is clarity, correctness, and a clean base for later decoding work.



## Block 1 — Load the binary file and validate the first ASTERIX header

This first block loads the binary stream into memory and performs a **minimal header sanity check**.

### Theory behind this block

At the file level, ASTERIX is not a text format and not a line-based format. It is a **binary stream**. The parser must therefore read bytes directly.

The first three octets already tell us a lot:

- `CAT` identifies **which category specification** must be used later.
- `LEN` tells us **how many bytes belong to the current block**.
- Because `LEN` includes the full message size, it is the value that lets us move safely from one message to the next.

At this stage we are **not decoding the payload yet**. We are only confirming that the binary file is readable and that the first header looks like a plausible ASTERIX message header.

### Data structure produced by this block

- `raw_data` → the complete binary file as `bytes`
- `raw_view` → a `memoryview` of the same bytes for efficient slicing later

### Expected output

A short summary showing:

- the file path used
- total number of bytes loaded
- the first `CAT`
- the first `LEN`
- a small hexadecimal preview of the stream


In [1]:

from pathlib import Path
import pandas as pd


def first_existing_path(candidates):
    """Return the first existing path from a list of candidates."""
    for candidate in map(Path, candidates):
        if candidate.exists():
            return candidate
    return None


# Update these candidates to match your local project structure.
BINARY_CANDIDATES = [
    Path("raw_data/datos_asterix_adsb.ast"),
]

BINARY_PATH = first_existing_path(BINARY_CANDIDATES)
if BINARY_PATH is None:
    raise FileNotFoundError(
        "No ASTERIX binary file was found. "
        "Update BINARY_CANDIDATES or set BINARY_PATH directly."
    )

# Load the full binary stream once.
raw_data = BINARY_PATH.read_bytes()
raw_view = memoryview(raw_data)

# Extract the first ASTERIX header if enough bytes are available.
if len(raw_data) < 3:
    raise ValueError("The file is too short to contain a valid ASTERIX header.")

first_cat = raw_data[0]
first_len = int.from_bytes(raw_data[1:3], byteorder="big")
first_bytes_hex = raw_data[:24].hex(" ").upper()

file_summary = pd.DataFrame(
    [
        {
            "file": str(BINARY_PATH),
            "size_bytes": len(raw_data),
            "first_cat": first_cat,
            "first_len": first_len,
        }
    ]
)

print(f"Loaded binary file: {BINARY_PATH}")
print(f"Total bytes: {len(raw_data):,}")
print(f"First header -> CAT={first_cat} | LEN={first_len}")
print(f"First 24 bytes (hex): {first_bytes_hex}")


Loaded binary file: raw_data/datos_asterix_adsb.ast
Total bytes: 86,455,774
First header -> CAT=21 | LEN=90
First 24 bytes (hex): 15 00 5A F3 1B 73 6B D3 A7 04 14 DB 01 01 00 03 C3 01 0F 04 B5 28 00 E4



## Block 2 — Split the binary stream into ASTERIX message blocks

This block walks through the file and separates it into **complete ASTERIX messages** using the header rule:

- read `CAT` from octet 1
- read `LEN` from octets 2-3
- slice exactly `LEN` bytes
- jump to the next block

### Theory behind this block

An ASTERIX `.ast` file does **not** need separators between messages. The parser advances using the declared block length. This is why `LEN` is so important: it is the only reliable way to know where the next message starts.

`LEN` is the **total length of the message block**, which means:

```text
message_bytes = [CAT][LEN][record area]
```

So, for every message:

- `message_bytes[:3]` is the block header
- `message_bytes[3:]` is the record area

### Why this block matters before any decoding

Before looking at FSPEC, UAP, or Data Items, we need a clean table of message boundaries. If the message slicing is wrong, every later decoding step will also be wrong.

### Data structures produced by this block

- `messages` → list of dictionaries, one dictionary per message
- `messages_df` → a compact pandas table summarizing the message boundaries

### Expected output

A concise summary with:

- total number of ASTERIX messages found
- count of messages per category
- a small preview table with offsets, categories, and lengths


In [2]:

def split_asterix_messages(raw_bytes: bytes):
    """Split a binary ASTERIX stream into message dictionaries."""
    stream = memoryview(raw_bytes)
    total_size = len(stream)
    messages = []
    offset = 0
    message_id = 1

    while offset < total_size:
        # Stop cleanly if fewer than 3 bytes remain.
        if offset + 3 > total_size:
            break

        cat = stream[offset]
        length = int.from_bytes(stream[offset + 1: offset + 3], byteorder="big")

        # Reject impossible message lengths.
        if length < 3:
            raise ValueError(f"Invalid LEN={length} at offset {offset}.")
        if offset + length > total_size:
            raise ValueError(
                f"Message at offset {offset} declares LEN={length}, "
                f"which exceeds the file size."
            )

        block = bytes(stream[offset: offset + length])
        record_area = block[3:]

        messages.append(
            {
                "message_id": message_id,
                "offset": offset,
                "cat": cat,
                "length": length,
                "payload_len": length - 3,
                "message_bytes": block,
                "record_area": record_area,
            }
        )

        offset += length
        message_id += 1

    return messages


messages = split_asterix_messages(raw_data)

messages_df = pd.DataFrame.from_records(
    [
        {
            "message_id": msg["message_id"],
            "offset": msg["offset"],
            "cat": msg["cat"],
            "length": msg["length"],
            "payload_len": msg["payload_len"],
        }
        for msg in messages
    ]
)

cat_counts = (
    messages_df["cat"]
    .value_counts()
    .sort_index()
    .rename_axis("CAT")
    .reset_index(name="messages")
)

print(f"ASTERIX messages found: {len(messages_df)}")
print("Messages per CAT:")
print(cat_counts.to_string(index=False))
print()
print("Message table preview:")
print(messages_df.head(8).to_string(index=False))


ASTERIX messages found: 975101
Messages per CAT:
 CAT  messages
  21    975101

Message table preview:
 message_id  offset  cat  length  payload_len
          1       0   21      90           87
          2      90   21      93           90
          3     183   21      90           87
          4     273   21      93           90
          5     366   21      93           90
          6     459   21      93           90
          7     552   21      61           58
          8     613   21      93           90



## Block 3 — Parse the FSPEC of the first record and list the announced FRNs

This block extracts the **FSPEC of the first record only** from each message.

### Theory behind this block

Inside each record, the FSPEC is the structure that announces **which fields are present**. It is not a value field; it is a **presence map**.

Each FSPEC octet works like this:

- the first 7 bits announce field presence (`FRN1`, `FRN2`, ..., `FRN7`, then `FRN8`, ... in later octets)
- the last bit is **FX**
- if `FX = 1`, another FSPEC octet follows
- if `FX = 0`, the FSPEC ends

So the FSPEC can be one octet long or several octets long.

### Why only the first record?

A message may contain multiple records, but after the FSPEC the record body contains items with different internal formats:

- **fixed length** → the number of octets is known immediately
- **variable / extended length** → an internal `FX` may extend the field
- **repetitive** → a repetition counter (`REP`) defines the size
- **compound** → a primary subfield announces which subfields follow

Because of that, the start of the next record cannot be found from the FSPEC alone. A correct parser would need to decode **all Data Fields of record #1** before it could safely locate **record #2**.

### Data structure produced by this block

- `fspec_df` → one row per message with:
  - message id
  - category
  - FSPEC in hex
  - FSPEC length in octets
  - list of FRNs detected in the first record

### Expected output

A concise table such as:

```text
Message 1 | CAT=21 | FSPEC=A0 | FRNs=FRN1, FRN3
```

This confirms that the message boundaries are correct and that the first-record presence map has been interpreted properly.


In [ ]:

def parse_first_record_fspec(record_area: bytes):
    """Parse the FSPEC found at the start of a record area.

    Returns
    -------
    fspec_octets : list[int]
        Raw FSPEC octets.
    frns : list[int]
        FRNs announced by the FSPEC, in increasing order.
    """
    if not record_area:
        return [], []

    fspec_octets = []
    frns = []
    frn_base = 1
    cursor = 0

    while cursor < len(record_area):
        octet = record_area[cursor]
        fspec_octets.append(octet)

        # Bits 8..2 map to seven consecutive FRNs.
        current_frns = list(range(frn_base, frn_base + 7))
        current_bits = [((octet >> bit_shift) & 1) for bit_shift in range(7, 0, -1)]
        frns.extend(frn for frn, is_present in zip(current_frns, current_bits) if is_present)

        # Bit 1 is FX.
        fx = octet & 0b1
        cursor += 1
        frn_base += 7

        if fx == 0:
            break

    return fspec_octets, frns


fspec_rows = []
for msg in messages:
    fspec_octets, frns = parse_first_record_fspec(msg["record_area"])
    fspec_rows.append(
        {
            "message_id": msg["message_id"],
            "cat": msg["cat"],
            "fspec_hex": " ".join(f"{octet:02X}" for octet in fspec_octets),
            "fspec_len": len(fspec_octets),
            "frns": frns,
            "frn_text": ", ".join(f"FRN{frn}" for frn in frns) if frns else "(none)",
        }
    )

fspec_df = pd.DataFrame.from_records(fspec_rows)

print(f"First-record FSPEC parsed for {len(fspec_df)} messages.")
print(
    fspec_df[["message_id", "cat", "fspec_hex", "frn_text"]]
    .head(10)
    .to_string(index=False)
)



## Block 4 — Map FRNs to Data Items using the category UAP tables

This block translates the FSPEC result into **human-readable Data Items**.

### Theory behind this block

The FSPEC tells us **which FRNs are present**, but it does **not** tell us directly which semantic field each FRN represents. That mapping is defined by the **User Application Profile (UAP)** of the category.

This is why the category matters:

- `FRN 1` in **CAT021** maps to one Data Item
- `FRN 1` in **CAT048** maps to another Data Item

So the real meaning of an FRN is always:

```text
(CAT, FRN) -> Data Item
```

### Why the UAP table is the correct place to look

The UAP is the category-specific table that answers questions such as:

- Which Data Item corresponds to `FRN 1`?
- Which items are fixed length?
- Which items are variable, repetitive, or compound?
- Which FRNs are reserved for `RE` and `SP`?

The **FSPEC only announces presence**. The **UAP gives meaning**.

### Pandas strategy used here

To keep the workflow clean and scalable, this block:

1. loads the CAT021 and CAT048 UAP tables once
2. normalizes them into a single lookup table `uap_df`
3. explodes the FRN lists from `fspec_df`
4. performs a pandas `merge` on `(CAT, FRN)`

This creates a tidy table where each row represents one detected field in one message.

### Important limitation

This block still maps only the **first record** of each message, because that is what the FSPEC block extracted. It does **not** yet decode field values and does **not** yet walk into later records.

### Data structures produced by this block

- `uap_df` → normalized UAP lookup table
- `data_items_df` → one row per detected `(message, CAT, FRN, Data Item)` pair
- `message_items_df` → one summary row per message

### Expected output

A compact per-message summary such as:

```text
Message 1 | CAT=21 | I021/010, I021/161
```

This is the final proof that the notebook correctly connected:

- message slicing
- first-record FSPEC parsing
- category-specific UAP mapping


In [ ]:

def read_uap_source(path: Path, sheet_name=None) -> pd.DataFrame:
    """Read either a CSV or an Excel UAP source."""
    if path.suffix.lower() == ".csv":
        return pd.read_csv(path)
    if path.suffix.lower() in {".xlsx", ".xls"}:
        return pd.read_excel(path, sheet_name=sheet_name)
    raise ValueError(f"Unsupported UAP file format: {path}")


def normalize_uap_table(df: pd.DataFrame, cat: int) -> pd.DataFrame:
    """Normalize a UAP table to a clean CAT/FRN/Data Item schema."""
    normalized = df.rename(
        columns={
            "Information": "Description",
            "Length": "Length in Octets",
        }
    ).copy()

    normalized["FRN_text"] = normalized["FRN"].astype(str).str.strip().str.upper()
    normalized = normalized[~normalized["FRN_text"].isin(["FX", "-", "N.A.", "NAN"])]
    normalized = normalized[normalized["FRN_text"].str.fullmatch(r"\d+")]

    normalized["CAT"] = cat
    normalized["FRN"] = normalized["FRN_text"].astype(int)
    normalized["Data Item"] = normalized["Data Item"].fillna("").astype(str).str.strip()
    normalized["Description"] = normalized["Description"].fillna("").astype(str).str.strip()
    normalized["Length in Octets"] = (
        normalized["Length in Octets"].fillna("").astype(str).str.strip()
    )

    return normalized[["CAT", "FRN", "Data Item", "Description", "Length in Octets"]]


def load_uap_table(cat: int, candidates, sheet_name=None) -> pd.DataFrame:
    """Load the first available UAP source for one category."""
    source_path = first_existing_path(candidates)
    if source_path is None:
        raise FileNotFoundError(
            f"No UAP file found for CAT{cat:03d}. Update the candidate paths."
        )
    raw_table = read_uap_source(source_path, sheet_name=sheet_name)
    return normalize_uap_table(raw_table, cat=cat)


uap_sources = {
    21: {
        "candidates": [
            Path("asterix_decoder/uap_tables/CAT021.csv"),
        ],
        "sheet_name": "CAT021_UAP",
    },
    48: {
        "candidates": [
            Path("asterix_decoder/uap_tables/CAT048.csv"),
        ],
        "sheet_name": "CAT048_UAP",
    },
}

uap_df = pd.concat(
    [
        load_uap_table(cat, cfg["candidates"], cfg["sheet_name"])
        for cat, cfg in uap_sources.items()
    ],
    ignore_index=True,
)

message_frn_df = (
    fspec_df.rename(columns={"cat": "CAT"})[["message_id", "CAT", "frns"]]
    .explode("frns")
    .rename(columns={"frns": "FRN"})
    .dropna(subset=["FRN"])
)
message_frn_df["FRN"] = message_frn_df["FRN"].astype(int)

# Join the detected FRNs with the UAP lookup.
data_items_df = (
    message_frn_df
    .merge(uap_df, on=["CAT", "FRN"], how="left")
    .sort_values(["message_id", "FRN"], kind="stable")
    .reset_index(drop=True)
)

unknown_mask = data_items_df["Data Item"].isna() | data_items_df["Data Item"].eq("")
data_items_df.loc[unknown_mask, "Data Item"] = (
    data_items_df.loc[unknown_mask]
    .apply(lambda row: f"I{int(row['CAT']):03d}/FRN{int(row['FRN'])}", axis=1)
)
data_items_df["Description"] = data_items_df["Description"].fillna("(unknown)")

message_items_df = (
    data_items_df.assign(item_label=data_items_df["Data Item"])
    .groupby(["message_id", "CAT"], as_index=False)
    .agg(
        item_count=("FRN", "size"),
        data_items=("item_label", lambda values: ", ".join(values)),
    )
    .sort_values("message_id", kind="stable")
    .reset_index(drop=True)
)

print(f"UAP rows loaded: {len(uap_df)}")
print(f"Detected Data Item rows: {len(data_items_df)}")
print()
print("Per-message Data Item summary:")
print(message_items_df.head(10).to_string(index=False))
